In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

employee = pd.read_csv('../data/employee_master.csv')
bench = pd.read_csv('../data/bench_allocation.csv')
finance = pd.read_csv('../data/finance_cost.csv')

df = employee.merge(bench, on='employee_id').merge(finance, on='employee_id')

le_dept = LabelEncoder()
le_perf = LabelEncoder()
df['department_enc'] = le_dept.fit_transform(df['department'])
df['performance_enc'] = le_perf.fit_transform(df['performance_rating'])

features = ['bench_days', 'tenure_years', 'salary', 'department_enc', 'performance_enc']
X = df[features]
y = df['attrition_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(random_state=42, class_weight='balanced')
log_model.fit(X_train_scaled, y_train)
log_prob = log_model.predict_proba(X_test_scaled)[:, 1]
log_auc = roc_auc_score(y_test, log_prob)

rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_prob)

if rf_auc > log_auc:
    best_model_name = "Random Forest"
else:
    best_model_name = "Logistic Regression"

print(f"Best model: {best_model_name}")

Best model: Logistic Regression


In [3]:
df_test = df.loc[X_test.index].copy()
df_test['attrition_prob'] = log_prob if best_model_name == "Logistic Regression" else rf_prob

df_test['replacement_cost'] = 0.75 * df_test['salary']
df_test['expected_loss'] = df_test['attrition_prob'] * df_test['replacement_cost']

intervention_map = {0: 50000, 1: 100000, 2: 150000}
df_test['intervention_cost'] = df_test['performance_enc'].map(intervention_map)

df_test['roi'] = df_test['expected_loss'] / df_test['intervention_cost']
df_test['recommend_intervention'] = df_test['roi'] > 6

df_test['recommend_intervention'].value_counts()

recommend_intervention
False    155
True      45
Name: count, dtype: int64

In [4]:
decision_table = df_test[['employee_id', 'department', 'attrition_prob',
                            'replacement_cost', 'expected_loss',
                            'intervention_cost', 'roi', 'recommend_intervention']] \
                            .sort_values('roi', ascending=False)

decision_table.to_csv('../data/decision_layer_output.csv', index=False)
decision_table.head(20)

,employee_id,department,attrition_prob,replacement_cost,expected_loss,intervention_cost,roi,recommend_intervention
772,EMP1772,Engineering,0.590838,1196848.50,707144.021666,50000,14.142880,True
709,EMP1709,Sales,0.790072,876357.75,692385.449515,50000,13.847709,True
990,EMP1990,Consulting,0.634863,1021974.00,648813.570466,50000,12.976271,True
424,EMP1424,Engineering,0.584202,1027242.75,600117.181369,50000,12.002344,True
519,EMP1519,Consulting,0.518516,1157010.75,599928.668796,50000,11.998573,True
175,EMP1175,Consulting,0.619811,961099.50,595700.217909,50000,11.914004,True
750,EMP1750,Engineering,0.566313,951225.00,538690.779277,50000,10.773816,True
870,EMP1870,Consulting,0.517309,1006013.25,520419.245575,50000,10.408385,True
318,EMP1318,Finance,0.416504,1192251.75,496577.792816,50000,9.931556,True
365,EMP1365,Finance,0.414720,1127295.75,467512.344375,50000,9.350247,True
